In [1]:
import pandas as pd

from protest_impact.util import project_root

count_df = pd.read_csv(
    project_root / "data" / "protest" / "protest_and_topic_counts.csv"
)
count_df["date"] = pd.to_datetime(count_df["date"])
count_df.sample(10)

,date,topic,count,type,media_id,name,keyword
766595,2021-02-14,covid,1,general,379335.0,come-on.de,corona
391853,2019-06-11,anti-war,1,general,385646.0,Lokale Nachrichten aus dem Landkreis Bad Kissi...,krieg
703667,2020-11-28,yellow jackets,1,general,385538.0,http://www.lz.de/,rente*
1288005,2022-11-26,covid,1,general,385644.0,Nachrichten aus Amberg - Onetz,corona
1067726,2022-03-16,environment,1,general,39005.0,merkur-online,naturschutz
514841,2020-03-23,covid,1,general,121272.0,abendzeitung-muenchen.de,quarantäne
1019164,2022-01-15,covid,1,general,39267.0,hna,corona
1269438,2022-11-06,lgbtq,1,general,119705.0,kreiszeitung.de,schwul
407653,2019-07-18,racism,1,general,385644.0,Nachrichten aus Amberg - Onetz,schwarz*
1088669,2022-04-07,refugees,1,general,39267.0,hna,behinderung


In [2]:
protests = count_df[count_df["type"] == "protest"]
protests.sample(10)

,date,topic,count,type,media_id,name,keyword
6588,2019-03-27,right wing,1,protest,NaN,NaN,NaN
1130,2015-09-07,refugees,1,protest,NaN,NaN,NaN
7753,2019-09-14,feminism,1,protest,NaN,NaN,NaN
6881,2019-05-14,climate,1,protest,NaN,NaN,NaN
1593,2016-03-11,labour,1,protest,NaN,NaN,NaN
528,2015-03-16,anti-war,1,protest,NaN,NaN,NaN
7556,2019-08-16,labour,1,protest,NaN,NaN,NaN
11561,2020-11-28,international,1,protest,NaN,NaN,NaN
10058,2020-06-22,anti-war,1,protest,NaN,NaN,NaN
4658,2018-05-09,environment,1,protest,NaN,NaN,NaN


In [3]:
from datetime import date

from dateutil.relativedelta import relativedelta

from protest_impact.data.news.sources.mediacloud import get_article_counts


def get_coverage_change(protest_event, interval):
    protest_date = protest_event["date"]
    protest_event["pre_coverage_start"] = (protest_date - interval).isoformat()
    protest_event["pre_coverage_end"] = protest_date.isoformat()
    protest_event["post_coverage_start"] = protest_date.isoformat()
    protest_event["post_coverage_end"] = (protest_date + interval).isoformat()
    protest_event["pre_coverage"] = count_df[
        (count_df["date"] >= protest_event["pre_coverage_start"])
        & (count_df["date"] < protest_event["pre_coverage_end"])
        & (count_df["type"] == "general")
        & (count_df["topic"] == protest_event["topic"])
    ].sum()["count"]
    protest_event["post_coverage"] = count_df[
        (count_df["date"] >= protest_event["post_coverage_start"])
        & (count_df["date"] < protest_event["post_coverage_end"])
        & (count_df["type"] == "general")
        & (count_df["topic"] == protest_event["topic"])
    ].sum()["count"]
    overall_coverage = get_article_counts(protest_event["media_id"])
    return overall_coverage
    protest_event["pre_coverage_overall"] = overall_coverage[
        protest_event["pre_coverage_start"] : protest_event["pre_coverage_end"]
    ].sum()

    return protest_event

In [4]:
get_coverage_change(protests.iloc[0].copy(), relativedelta(months=1))

/var/folders/6v/w9nn6c_n4qdbrjwfnq7695n00000gn/T/ipykernel_34383/851249838.py:16: FutureWarning: The default value of numeric_only in DataFrame.sum is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  ].sum()["count"]
/var/folders/6v/w9nn6c_n4qdbrjwfnq7695n00000gn/T/ipykernel_34383/851249838.py:22: FutureWarning: The default value of numeric_only in DataFrame.sum is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  ].sum()["count"]


{'error': 'Error(s): User exceeded weekly requests limit of 10000. You have exceeded your quota of requests or stories. Please contact\ninfo@mediacloud.org with quota questions.\n'}

In [1]:
from os import environ

from dotenv import load_dotenv

from protest_impact.util.cache import get

load_dotenv()

response = get(
    # "https://api.mediacloud.org/api/v2/auth/profile",
    "https://api.mediacloud.org/api/v2/stats/list",
    # headers={"Accept": "application/json"},
    params={
        "key": environ["MEDIACLOUD_API_KEY"],
    },
)
result = response.json()
result

{'error': 'Error(s): User exceeded weekly requests limit of 10000. You have exceeded your quota of requests or stories. Please contact\ninfo@mediacloud.org with quota questions.\n'}